# Explore the visual baseline dataset
Read completed outputs from `outputs/visual_baseline/`. This notebook does not extract videos,
recompute embeddings, or overwrite JSON files. Run cells in order with the project `.venv` kernel.


In [ ]:
import sys
from pathlib import Path

import numpy as np
from IPython.display import HTML, display

# Support kernels started from the repository root or the notebooks folder.
ROOT = next((path for path in (Path.cwd(), *Path.cwd().parents)
             if (path / "src").is_dir() and (path / "main.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("Start this notebook inside the repository")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.inspection import load_recordings, load_embedding, detection_rows, table_html

OUTPUT_DIR = ROOT / "outputs/visual_baseline"
RAW_DIR = ROOT / "data/raw/reassemble"
recordings = load_recordings(OUTPUT_DIR)
print("Available recordings:", *recordings, sep="\n")


## Dataset coverage
Window counts and represented segments are different; overlapping windows are not independent shots.


In [ ]:
from src.dataset_report import summarize_dataset

all_clips = [clip for item in recordings.values() for clip in item["clips"]]
report = summarize_dataset(all_clips)
print("Recordings:", report["recording_count"], "Clips:", report["clip_count"])
for field in ("windows_by_class", "segments_by_class"):
    print(field)
    labels = sorted({label for row in report["recordings"].values() for label in row[field]})
    rows = [{"recording_id": name, **{label: row[field].get(label, 0) for label in labels}}
            for name, row in report["recordings"].items()]
    display(HTML(table_html(rows)))


## Select a recording


In [ ]:
# Change this ID to inspect another completed recording.
VIDEO_ID = next(iter(recordings))
recording = recordings[VIDEO_ID]
manifest = recording["manifest"]
clips = recording["clips"]
display(manifest)
print("Clips:", len(clips))


In [ ]:
columns = ["clip_id", "segment_id", "ground_truth", "object", "success",
           "start_frame", "end_frame_exclusive", "original_start_time",
           "original_end_time_exclusive"]
display(HTML(table_html(clips[:20], columns)))


## Compare original annotations and saved labels
Checks run in memory; no labeled JSON is written. Frame counts do not prove synchronization.


In [ ]:
from src.labeling import load_annotations, label_clips, validate_video

h5_path = RAW_DIR / manifest["source"]["relative_path"]
video_path = recording["directory"] / f"{VIDEO_ID}.mp4"
annotations, timestamps = load_annotations(h5_path, manifest["camera"])
fps, frame_count = validate_video(video_path, timestamps)
display(HTML(table_html(annotations)))
print("FPS:", fps, "Frames:", frame_count)
print("Timestamp spacing (min, median, max):",
      np.min(np.diff(timestamps)), np.median(np.diff(timestamps)), np.max(np.diff(timestamps)))
recomputed = label_clips(h5_path, video_path, clips, manifest["camera"])
fields = ("ground_truth", "segment_id", "object", "success")
assert all(all(old[key] == new[key] for key in fields)
           for old, new in zip(clips, recomputed))
print("Saved labels match the original annotations.")


### Optional YOLO detections
`object` is the object from the activity annotation, not a detected object.
`detections` stores sampled frames as `{"timestamp": relative_video_seconds, "objects": [...]}`.
Each object has `label`, `confidence` and `bbox` (`[x1, y1, x2, y2]` in pixels).
An empty list is supported. If YOLO is disabled, it does **not** mean that the scene contains no objects.
To generate detections later, enable `USE_YOLO` in `main.py` and select a new output directory;
existing recordings are not enriched automatically. No YOLO model is loaded by these notebooks.


In [ ]:
detection_columns = ["recording_id", "clip_id", "timestamp", "label", "confidence", "bbox"]
detections = detection_rows(clips)
print("YOLO enabled for this run:", manifest["config"]["use_yolo"])
print("Stored sampled frames:", sum(len(c.get("detections", []) or []) for c in clips))
print("Stored object detections:", len(detections))
display(HTML(table_html(detections[:20], detection_columns)))


## Preview a sampled frame
Change `CLIP_INDEX` and `SAMPLE_INDEX`. Boxes appear only for detections stored at this frame time.


In [ ]:
import cv2
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

CLIP_INDEX = 0
SAMPLE_INDEX = 7
clip = clips[CLIP_INDEX]
frame_index = clip["sampled_frame_indices"][SAMPLE_INDEX]
frame_time = clip["frame_times"][SAMPLE_INDEX]
cap = cv2.VideoCapture(str(video_path))
try:
    cap.set(cv2.CAP_PROP_POS_FRAMES, frame_index)
    ok, frame = cap.read()
finally:
    cap.release()
if not ok:
    raise RuntimeError(f"Cannot read frame {frame_index}")
fig, ax = plt.subplots(figsize=(10, 6))
ax.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
for row in detection_rows([clip]):
    if np.isclose(row["timestamp"], frame_time, rtol=0, atol=1e-6):
        x1, y1, x2, y2 = row["bbox"]
        ax.add_patch(Rectangle((x1, y1), x2-x1, y2-y1, fill=False, edgecolor="lime"))
        ax.text(x1, y1, f"{row['label']} {row['confidence']:.2f}", color="lime")
ax.set_title(f"{clip['clip_id']} | ground truth: {clip['ground_truth']} | frame {frame_index}")
ax.axis("off")
plt.show()
